# 11 - 工具类参考

> **关联**: 本文演示 sqlseed 内部工具类的使用，包括 MetricsCollector、sql_safe、Progress、Logger 和 schema_helpers。

## 你将学到

- MetricsCollector 性能度量
- sql_safe SQL 注入防护
- Progress 多后端进度系统
- Logger structlog 日志
- schema_helpers AUTOINCREMENT 检测

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| 08 | MCP 服务器集成 | Plugins: MCP | 07 |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| **→ 11** | **工具类参考** | **Utils** | **01** |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| 内部工具集 | `src/sqlseed/_utils/` | `progress.py`, `sql_safe.py`, `metrics.py` |

## 1. MetricsCollector 性能度量

MetricsCollector 收集和汇总性能指标，支持按名称过滤和统计。

In [2]:
from sqlseed._utils.metrics import MetricsCollector

metrics = MetricsCollector()

metrics.record("fill_time", 1.234)
metrics.record("fill_time", 0.567)
metrics.record("fill_time", 2.891)
metrics.record("insert_count", 1000)
metrics.record("insert_count", 2000)

print("MetricsCollector API:")
print("  record('fill_time', 1.234) → 记录指标")
print(f"  get_entries('fill_time') → {len(metrics.get_entries('fill_time'))} 条")
print(f"  get_entries() → {len(metrics.get_entries())} 条 (全部)")

summary = metrics.summary()
print("\n  summary():")
for name, stats in summary.items():
    print(f"    {name}: count={stats['count']}, avg={stats['avg']:.3f}, min={stats['min']:.3f}, max={stats['max']:.3f}")

metrics.clear()
print(f"\n  clear() → {len(metrics.get_entries())} 条 (已清空)")

MetricsCollector API:
  record('fill_time', 1.234) → 记录指标
  get_entries('fill_time') → 3 条
  get_entries() → 5 条 (全部)

  summary():
    fill_time: count=3, avg=1.564, min=0.567, max=2.891
    insert_count: count=2, avg=1500.000, min=1000.000, max=2000.000

  clear() → 0 条 (已清空)


## 2. sql_safe SQL 注入防护

sql_safe 模块提供 SQL 标识符转义和验证功能，防止 SQL 注入。

In [3]:
from sqlseed._utils.sql_safe import build_insert_sql, quote_identifier, validate_table_name

print("sql_safe — SQL 注入防护:\n")

print(f"  quote_identifier('table_name') → {quote_identifier('table_name')}")
print("  quote_identifier('table\"name') → " + quote_identifier('table\"name'))

print(f"\n  validate_table_name('users') → {validate_table_name('users')}")

sql = build_insert_sql("users", ["name", "email", "age"])
print("\n  build_insert_sql('users', ['name', 'email', 'age']):")
print(f"    {sql}")

sql_safe — SQL 注入防护:

  quote_identifier('table_name') → "table_name"
  quote_identifier('table"name') → "table""name"

  validate_table_name('users') → "users"

  build_insert_sql('users', ['name', 'email', 'age']):
    INSERT INTO "users" ("name", "email", "age") VALUES (?, ?, ?)


## 3. Progress 多后端进度系统

sqlseed 使用 Strategy Pattern 实现跨环境进度条:
- **终端**: `RichProgressBackend` (基于 Rich)
- **Jupyter**: `TqdmNotebookBackend` (基于 tqdm.auto)
- **无 UI**: `NullProgressBackend` (零开销)

`create_progress()` 会自动检测运行环境并选择合适的后端。

In [4]:
from sqlseed._utils.progress import create_progress

# create_progress() returns a multi-backend Progress context manager
# Used internally by fill_table for batch progress display
progress = create_progress()
print(f'Progress type: {type(progress).__name__}')
print()
print('Usage in fill_table:')
print('  with create_progress() as progress:')
print('      task = progress.add_task("Filling...", total=count)')
print('      for batch in data_stream.generate():')
print('          progress.update(task, advance=len(batch))')
print()
print('Columns: Spinner, Bar, Percentage, Count, Speed, Time Remaining')

Progress type: TqdmNotebookBackend

Usage in fill_table:
  with create_progress() as progress:
      task = progress.add_task("Filling...", total=count)
      for batch in data_stream.generate():
          progress.update(task, advance=len(batch))

Columns: Spinner, Bar, Percentage, Count, Speed, Time Remaining


## 4. Logger structlog 日志

sqlseed 使用 structlog 进行结构化日志记录，通过 `GeneratorConfig.log_level` 控制日志级别。

In [5]:
from sqlseed.config.models import GeneratorConfig

print("Logger — structlog 日志:")
print("  sqlseed 使用 structlog 进行结构化日志记录")
print("  通过 GeneratorConfig.log_level 控制日志级别")
print("  默认: INFO")

config = GeneratorConfig(db_path=str(db_path), log_level="DEBUG")
print(f"\n  GeneratorConfig(log_level='DEBUG') → {config.log_level}")

Logger — structlog 日志:
  sqlseed 使用 structlog 进行结构化日志记录
  通过 GeneratorConfig.log_level 控制日志级别
  默认: INFO

  GeneratorConfig(log_level='DEBUG') → DEBUG


## 5. schema_helpers AUTOINCREMENT 检测

schema_helpers 提供 AUTOINCREMENT 主键检测功能，用于 ColumnMapper Level 1 判断是否跳过生成。

In [6]:

print("schema_helpers — AUTOINCREMENT 检测:")
print("  detect_autoincrement(execute_fn, table_name, column_name) → 检测列是否为自增主键")
print("  用于 ColumnMapper Level 1 判断是否跳过生成")

with sqlseed.connect(str(db_path)) as orch:
    col_info = orch.get_column_info("organizations")
    for col in col_info:
        if col.is_primary_key:
            print(f"\n  {col.name}: is_pk={col.is_primary_key}, is_autoincrement={col.is_autoincrement}")

schema_helpers — AUTOINCREMENT 检测:
  detect_autoincrement(execute_fn, table_name, column_name) → 检测列是否为自增主键
  用于 ColumnMapper Level 1 判断是否跳过生成

  org_code: is_pk=True, is_autoincrement=False


## 6. ExpressionEngine 表达式引擎

ExpressionEngine 使用 simpleeval 执行安全的 Python 表达式，提供 21 个白名单函数。

In [7]:
from sqlseed.core.expression import ExpressionEngine

engine = ExpressionEngine()

# Basic arithmetic
print(f"2 + 3 = {engine.evaluate('a + b', {'a': 2, 'b': 3})}")

# String operations
print(f"upper: {engine.evaluate('name.upper()', {'name': 'hello'})}")

# Safe functions (21 available)
print(f"abs(-5): {engine.evaluate('abs(x)', {'x': -5})}")
print(f"len: {engine.evaluate('len(s)', {'s': 'abc'})}")
print(f"min(1,2,3): {engine.evaluate('min(a,b,c)', {'a': 1, 'b': 2, 'c': 3})}")

2 + 3 = 5
upper: HELLO
abs(-5): 5
len: 3
min(1,2,3): 1


## ✅ 总结

| 工具 | 功能 | 状态 |
|---|---|---|
| MetricsCollector | 性能度量 | ✅ |
| sql_safe | SQL 注入防护 | ✅ |
| Progress | 多后端进度系统 | ✅ |
| Logger | structlog 日志 | ✅ |
| schema_helpers | AUTOINCREMENT 检测 | ✅ |

**下一步**: [12-testing-patterns.ipynb](12-testing-patterns.ipynb) — 测试集成模式

In [8]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
